# 🧬 Hydra-PaiNN: Aqueous Tautomer Ratio & Free Energy Predictor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeantChen/TautomerRatio/blob/main/notebooks/Hydra_PaiNN_Quickstart.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue.svg?logo=github)](https://github.com/KeantChen/TautomerRatio)

**Hydra-PaiNN** is a physics-informed multi-tower delta-learning framework with an upfront 4-tier CBH-2 topological triage for predicting aqueous tautomer equilibrium constants ($K_\text{eq}$), population percentages, and relative free energies ($\Delta G_\text{aq}$).

### 🚀 Key Capabilities:
- **Upfront 2D Topological Triage**: 4-Tier confidence hierarchy based on aromaticity conservation and Raghavachari CBH-2 homodesmotic rules.
  - **Tier 1 (Golden Zone)**: Expected MAE $\approx 1.29$ kcal/mol (Chemical accuracy regime).
  - **Tier 2 (Aromatic Scaffold)**: Expected MAE $\approx 2.12$ kcal/mol.
  - **Tier 3 (Flexible Aliphatic)**: Expected MAE $\approx 2.37$ kcal/mol.
  - **Tier 4 (Risk Zone)**: Expected MAE $\approx 3.79$ kcal/mol.
- **Physics-Informed Multi-Tower Delta-Learning**: Equivariant PaiNN neural network correcting semiempirical GFN2-xTB / ALPB water solvation physics.
- **Microsecond Triage / Seconds Inference**: Instant 2D screening + fast 3D-quantum-ML ratio prediction.


In [ ]:
#@title 1. Environment Setup (Clone Repo & Install Dependencies)
#@markdown Run this cell once to download the model, install dependencies (RDKit, PyG), and configure Linux xTB.
import os, sys

# 1. Clone repository into Colab workspace
if not os.path.exists('/content/TautomerRatio'):
    print('📥 Cloning Hydra-PaiNN repository from GitHub...')
    !git clone https://github.com/KeantChen/TautomerRatio.git /content/TautomerRatio

%cd /content/TautomerRatio

# 2. Install Python dependencies
print('📦 Installing RDKit and PyTorch Geometric...')
!pip install -q rdkit torch_geometric

# 3. Download and set up Linux xTB binary
if not os.path.exists('/content/TautomerRatio/bin/xtb'):
    print('⚙️ Downloading precompiled xTB 6.7.1 for Linux x86_64...')
    !wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz
    !tar -xf xtb-6.7.1-linux-x86_64.tar.xz
    !cp xtb-6.7.1/bin/xtb bin/xtb
    !chmod +x bin/xtb

# 4. Configure environment variables for xTB runtime
os.environ['XTBPATH'] = '/content/TautomerRatio/xtb-6.7.1/share/xtb'
os.environ['PATH'] = '/content/TautomerRatio/xtb-6.7.1/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/content/TautomerRatio/xtb-6.7.1/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

print('✅ Environment setup successfully completed! You are ready to run predictions.')


In [ ]:
#@title 2. Verify Installation & Model Loading
#@markdown Tests that PyTorch (CUDA/CPU), RDKit, Scalers, Model Weights, and xTB are properly functioning.
!python verify_installation.py


In [ ]:
#@title 3. Interactive Tautomer Prediction & 2D Structure Visualization
#@markdown Enter two tautomeric SMILES representations and simulation parameters:
smiles_A = 'Oc1ccccn1'  #@param {type:'string'}
smiles_B = 'O=c1cccc[nH]1'  #@param {type:'string'}
solvent = 'water'  #@param ['water', 'gas']
temperature_K = 298.15  #@param {type:'number'}

from IPython.display import display, HTML
from rdkit import Chem
from rdkit.Chem import Draw
from tautomer_predictor import TautomerPredictor

predictor = TautomerPredictor()

print(f'🔬 Running Hydra-PaiNN prediction for:')
print(f'   Tautomer A: {smiles_A}')
print(f'   Tautomer B: {smiles_B}\n')

res = predictor.predict(smiles_A, smiles_B, temperature=temperature_K, solvent=solvent)
triage = res['triage']
pred = res['recommended_prediction']

# 2D Molecular Visualization
mol_A = Chem.MolFromSmiles(smiles_A)
mol_B = Chem.MolFromSmiles(smiles_B)
legends = [
    f"Tautomer A ({pred['population_pct']['A']:.1f}%)" + (" [Major]" if pred['major_tautomer'] == 'A' else ''),
    f"Tautomer B ({pred['population_pct']['B']:.1f}%)" + (" [Major]" if pred['major_tautomer'] == 'B' else '')
]
img = Draw.MolsToGridImage([mol_A, mol_B], molsPerRow=2, subImgSize=(320, 200), legends=legends)
display(img)

# Visual Card
tier_colors = {1: '#10b981', 2: '#3b82f6', 3: '#f59e0b', 4: '#ef4444'}
tier_color = tier_colors.get(triage['tier'], '#6b7280')

html_card = f"""
<div style='background-color: #f8fafc; border: 1px solid #e2e8f0; border-radius: 12px; padding: 20px; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; max-width: 650px; margin-top: 15px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);'>
  <div style='display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #e2e8f0; padding-bottom: 12px; margin-bottom: 16px;'>
    <h3 style='margin: 0; color: #1e293b; font-size: 18px;'>Prediction Summary</h3>
    <span style='background-color: {tier_color}; color: white; padding: 4px 12px; border-radius: 9999px; font-size: 12px; font-weight: bold;'>
      Tier {triage['tier']} ({triage['confidence']})
    </span>
  </div>
  
  <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;'>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Relative Free Energy (ΔG)</div>
      <div style='font-size: 20px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['delta_G_kcal_mol']:.2f} <span style='font-size: 13px; font-weight: normal; color: #64748b;'>kcal/mol</span></div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Equilibrium Constant (K_eq)</div>
      <div style='font-size: 20px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['K_eq']:.4e}</div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Major Tautomer</div>
      <div style='font-size: 20px; font-weight: bold; color: #2563eb; margin-top: 4px;'>Tautomer {pred['major_tautomer']}</div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Ratio</div>
      <div style='font-size: 18px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['ratio']}</div>
    </div>
  </div>

  <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7; font-size: 13px; color: #334155; line-height: 1.5;'>
    <strong>Topological Triage:</strong> {triage['message']}<br>
    <strong>Expected Benchmark MAE:</strong> ±{triage['expected_mae']:.2f} kcal/mol
  </div>
</div>
"""
display(HTML(html_card))


In [ ]:
#@title 4. Batch Prediction on a CSV File
#@markdown Run predictions across multiple pairs from a CSV file (e.g. `examples/test_cases.csv` or your uploaded file).
import pandas as pd

!python predict.py --csv examples/test_cases.csv --output colab_predictions.csv

df_out = pd.read_csv('colab_predictions.csv')
print(f'✅ Batch prediction completed! Processed {len(df_out)} pairs.')
display(df_out)


In [ ]:
#@title 5. Ultra-Fast 2D Topological Triage Screening (Microsecond-scale)
#@markdown Screen thousands of tautomer pairs based on CBH-2 homodesmotic rules without requiring 3D quantum calculations.
!python predict.py --csv examples/test_cases.csv --triage_only
